In [1]:
import os, warnings
import numpy as np
import tensorflow as tf

SEED = 31415

def set_seed(seed=SEED):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

warnings.filterwarnings("ignore")
set_seed()

2025-02-04 14:19:17.471294: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738678757.493486    1371 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738678757.498949    1371 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-04 14:19:17.520444: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import matplotlib.pyplot as plt
from matplotlib import gridspec

plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('image', cmap='magma')

In [3]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

# Load training and validation sets
ds_train_ = image_dataset_from_directory(
    '../datasets/car-or-truck/train',
    labels='inferred',
    label_mode='binary',
    image_size=[128, 128],
    interpolation='nearest',
    batch_size=64,
    shuffle=True,
)

ds_valid_ = image_dataset_from_directory(
    '../datasets/car-or-truck/valid',
    labels='inferred',
    label_mode='binary',
    image_size=[128, 128],
    interpolation='nearest',
    batch_size=64,
    shuffle=False,
)

# Data Pipeline
def convert_to_float(image, label):
    image = tf.image.convert_image_dtype(image, dtype=tf.float32)
    return image, label

AUTOTUNE = tf.data.experimental.AUTOTUNE
ds_train = (
    ds_train_
    .map(convert_to_float)
    .cache()
    .prefetch(buffer_size=AUTOTUNE)
)
ds_valid = (
    ds_valid_
    .map(convert_to_float)
    .cache()
    .prefetch(buffer_size=AUTOTUNE)
)

Found 5117 files belonging to 2 classes.


2025-02-04 14:19:20.878504: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Found 5051 files belonging to 2 classes.


In [4]:
pretrained_base = tf.keras.models.load_model(
    '../resources/cv-course-models/inceptionv1/inceptionv1.h5'
)

ValueError: Unknown layer: 'KerasLayer'. Please ensure you are using a `keras.utils.custom_object_scope` and that this object is included in the scope. See https://www.tensorflow.org/guide/keras/save_and_serialize#registering_the_custom_object for details.

In [ ]:
# Decide whether this base should be trainable or not.
pretrained_base.trainable = False

In [ ]:
from tensorflow.keras import Sequential, layers

HIDDEN_SHAPE, OUTPUT_SHAPE = 6, 1
ReLU, SIGMOID = 'relu', 'sigmoid'

model = Sequential([
    pretrained_base,
    layers.Flatten(),
    # Attach a head of dense layers.
    layers.Dense(units=HIDDEN_SHAPE, activation=ReLU),
    layers.Dense(units=OUTPUT_SHAPE, activation=SIGMOID),
])

In [ ]:
# What loss function should you use for a binary classification problem?
optimizer = tf.keras.optimizers.Adam(epsilon=0.01)
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['binary_accuracy'],
)

In [ ]:
EPOCHS = 30

history = model.fit(
    ds_train,
    validation_data=ds_valid,
    epochs=EPOCHS,
)

In [ ]:
from IPython.display import display, Markdown

content = f"""
**Q**: **Do you notice a difference between these learning curves and the curves for VGG16 
from the tutorial? What does this difference tell you about what this model (InceptionV2) 
learned compared to VGG16? Are there ways in which one is better than the other? Worse?**<br>
**A**: The training loss and validation loss stay fairly close, which is evidence that the model
isn't just memorizing the training data, but rather learning general properties of the two classes.
But, because this model converges at a loss greater than the VGG16 model, it's likely that it is
underfitting some, and could benefit from some extra capacity.
"""

display(Markdown(content))